# 02 · From a clinical note to an OMOP table

`OmopExtractionPipeline` reads notes with an LLM, codes what it finds with the OMOP
vocabularies and returns rows of OMOP CDM tables.

This example uses the predefined **ECOG performance status** definition, which fills the
`measurement` table. It needs no terminology mapper: only the vocabulary database built by
`aatm init` (`.aatm/omop.db`) and an LLM (`OPENAI_API_KEY` in `.env`).

In [1]:
# Run from the folder where you ran `aatm init` (it holds .aatm/ and .env).
import os

if not os.path.isdir(".aatm") and os.path.isdir("../.aatm"):
    os.chdir("..")

## 1. Describe the notes

`ClinicalNote` holds the text and the values that come *with* the note (patient, date,
visit). The model never sees those: they go straight to their columns.

In [2]:
from aatm.omop import ClinicalNote, OmopExtractionPipeline

notes = [
    ClinicalNote(
        text="Assessment 05/12/2025\nOncology follow-up. Bedbound, ECOG 4.",
        person_id=3,
        note_id=107,
        note_date="2025-06-01",
        visit_occurrence_id=507,
    ),
    ClinicalNote(text="Fully active, no restrictions. PS 0.", person_id=4, note_date="2025-06-02"),
]

## 2. Run the pipeline

The pipeline is asynchronous: use `await` in a notebook, `asyncio.run(pipeline(notes))`
in a script.

In [3]:
pipeline = OmopExtractionPipeline("ecog_performance_status", model_id="openai/gpt-5-mini")

tables = await pipeline(notes)
tables["measurement"].dropna(axis=1, how="all")  # hide the empty CDM columns

,measurement_id,person_id,measurement_concept_id,measurement_date,measurement_type_concept_id,value_as_number,value_as_concept_id,visit_occurrence_id,measurement_source_value,measurement_source_concept_id,value_source_value
0,1,3,36305384,2025-05-12,32858,4.0,4174251,507,ECOG 4,4308014,4
1,2,4,36305384,2025-06-02,32858,0.0,4175026,<NA>,PS 0.,4308014,0


- `measurement_concept_id` **36305384** is *ECOG Performance Status score* (LOINC 89247-1).
- `measurement_date` is the date written in the note when there is one, else `note_date`.
- `measurement_type_concept_id` **32858** means *NLP*: derived from text.
- `value_as_number` / `value_as_concept_id` hold the grade as a number and as a concept.
- `*_source_value` columns keep the words of the note.

## 3. Inspect what happened

Every row can be traced back to the exact words it came from:

In [4]:
provenance_cols = ["measurement_id", "person_id", "_note_id", "_text", "_start", "_end"]
tables.to_dataframes(provenance=True)["measurement"][provenance_cols]

,measurement_id,person_id,_note_id,_text,_start,_end
0,1,3,107.0,ECOG 4,52,58
1,2,4,NaN,PS 0.,31,36


Columns that could not be filled are never silently dropped; they are listed as issues
(an empty table means none):

In [5]:
tables.issues_dataframe()

,table,row_id,note_index,note_id,entity_id,column,reason,detail


## Next

Write your own definition: [03_custom_entity_definition.ipynb](03_custom_entity_definition.ipynb).